## Import the library

In [1]:
from pyspark.sql import SparkSession

In [2]:

spark = SparkSession.builder \
    .appName("Read CSV") \
    .getOrCreate()

## Fact table

In [3]:
df_fact = spark.read.csv(
    "data/fact_transactions_Updated_.csv",
    header=True,
    inferSchema=True
)

In [4]:
df_fact.show(5)

+--------------+---------+--------+----------+---------+--------------------+------------+-----------+-------------------+----------------+-----------+-----------+--------------+--------------+------------------+
|transaction_id|worker_id| date_id|channel_id|market_id|    transaction_type|amount_local| amount_usd|transaction_outcome|is_fraud_flagged|is_disputed|is_reversed|velocity_score|fraud_loss_usd|processing_time_ms|
+--------------+---------+--------+----------+---------+--------------------+------------+-----------+-------------------+----------------+-----------+-----------+--------------+--------------+------------------+
|         val_0| val_2266|20241016|     val_2|    val_2|Wallet Funding vi...| 488.2946556|997.6967785|           Disputed|           false|      false|       true|   630.2743648|           0.0|               990|
|         val_1|   val_82|20230425|     val_5|    val_3|Cross-Border Remi...| 27.48144382|37.33027607|           Disputed|           false|       tr

## Dimension table

In [5]:
df_channel = spark.read.csv(
    "data/dim_channel.csv",
    header=True,
    inferSchema=True
)

In [6]:
df_channel.show(5)

+----------+---------------+--------------------+----------+--------------+-----------------+
|channel_id|   channel_type|     channel_subtype|is_digital|avg_fraud_rate|requires_internet|
+----------+---------------+--------------------+----------+--------------+-----------------+
|     val_0|API/Third-Party|        Merchant POS|      true|   857.6105854|             true|
|     val_1|            POS| USSD Agent Assisted|      true|   624.7460929|            false|
|     val_2|            POS|     API Integration|      true|   559.8086507|            false|
|     val_3|          Agent|Human Agent (Branch)|      true|   509.0999031|            false|
|     val_4|            POS|     API Integration|     false|    689.533128|             true|
+----------+---------------+--------------------+----------+--------------+-----------------+
only showing top 5 rows


In [7]:
df_date = spark.read.csv(
    "data/dim_date_updated.csv",
    header=True,
    inferSchema=True
)

In [8]:
df_date.show(5)

+--------+---------+----+-------+-------+-----------+----------+------------+-----------+
| date_id|full_date|year|  month|quarter|day_of_week|is_weekend|is_month_end|week_number|
+--------+---------+----+-------+-------+-----------+----------+------------+-----------+
|20230101| 1/1/2023|2023|January|     Q1|     Sunday|      true|       false|       NULL|
|20230102| 1/2/2023|2023|January|     Q1|     Monday|     false|       false|       NULL|
|20230103| 1/3/2023|2023|January|     Q1|    Tuesday|     false|       false|       NULL|
|20230104| 1/4/2023|2023|January|     Q1|  Wednesday|     false|       false|       NULL|
|20230105| 1/5/2023|2023|January|     Q1|   Thursday|     false|       false|       NULL|
+--------+---------+----+-------+-------+-----------+----------+------------+-----------+
only showing top 5 rows


In [9]:
df_market = spark.read.csv(
    "data/dim_market.csv",
    header=True,
    inferSchema=True 
)

In [10]:
df_market.show(5)

+---------+------------+--------+--------------+------------------+-----------------+-----------+------------------+
|market_id|     country|iso_code|local_currency|  dominant_channel|  regulatory_tier|usd_fx_rate|market_fraud_index|
+---------+------------+--------+--------------+------------------+-----------------+-----------+------------------+
|    val_0|     Nigeria|      NG|           NGN|Mobile Money Agent|Emerging (Tier 3)|80.23307561|       392.8462477|
|    val_1|       Kenya|      KE|           KES|        Mobile App|  Mature (Tier 1)|366.0779776|       267.5220342|
|    val_2|       Ghana|      GH|           GHS|     Bank Transfer|  Mature (Tier 1)|872.4207008|       421.6032103|
|    val_3|South Africa|      ZA|           ZAR|Mobile Money Agent|  Mature (Tier 1)|354.4362812|       797.5121313|
+---------+------------+--------+--------------+------------------+-----------------+-----------+------------------+



In [11]:
df_worker = spark.read.csv(
    "data/dim_worker.csv",
    header=True,
    inferSchema=True 
)

In [12]:
df_worker.show(5)

+---------+-------------+--------------------+-------------------+--------------------+----------+--------+-----------+---------+--------------------+
|worker_id|  worker_name|         gig_segment|account_tenure_days|            kyc_tier|    gender|age_band| risk_score|is_active|   preferred_channel|
+---------+-------------+--------------------+-------------------+--------------------+----------+--------+-----------+---------+--------------------+
|    val_0|    Ngozi Eze|      Delivery Rider|                618| Tier 0 (Unverified)|      Male|     55+|34.75549783|    false|                 Web|
|    val_1|Sipho Dlamini| Ride-Hailing Driver|                398|Tier 2 (ID Verified)|Non-Binary|   45-54|10.28072431|    false|               Agent|
|    val_2|   Nana Adjei|  On-Demand Handyman|                950|Tier 2 (ID Verified)|      Male|   35-44|30.20684661|     true|                 Web|
|    val_3| Mary Wairimu|Agricultural Day ...|               1210|Tier 2 (ID Verified)|Non-Bin

## Star schema

### Fact table is join on channel_id . This is join 1

In [13]:
# Left join: retain every record from the fact table
df_joined = df_fact.join(
    df_channel,
    df_fact["CHANNEL_ID"] == df_channel["CHANNEL_ID"],
    "left"
).drop(df_channel["CHANNEL_ID"])  # avoids duplicate CHANNEL_ID column

df_joined.show(5)

+--------------+---------+--------+----------+---------+--------------------+------------+-----------+-------------------+----------------+-----------+-----------+--------------+--------------+------------------+------------+-------------------+----------+--------------+-----------------+
|transaction_id|worker_id| date_id|channel_id|market_id|    transaction_type|amount_local| amount_usd|transaction_outcome|is_fraud_flagged|is_disputed|is_reversed|velocity_score|fraud_loss_usd|processing_time_ms|channel_type|    channel_subtype|is_digital|avg_fraud_rate|requires_internet|
+--------------+---------+--------+----------+---------+--------------------+------------+-----------+-------------------+----------------+-----------+-----------+--------------+--------------+------------------+------------+-------------------+----------+--------------+-----------------+
|         val_0| val_2266|20241016|     val_2|    val_2|Wallet Funding vi...| 488.2946556|997.6967785|           Disputed|        

In [14]:
# Left join: retain every record from the fact table
df_joined1 = df_joined.join(
    df_date,
    df_joined["date_id"] == df_date["date_id"],
    "left"
).drop(df_date["date_id"])  # avoids duplicate date_id column

In [15]:
df_joined1.show(5)

+--------------+---------+--------+----------+---------+--------------------+------------+-----------+-------------------+----------------+-----------+-----------+--------------+--------------+------------------+------------+-------------------+----------+--------------+-----------------+----------+----+---------+-------+-----------+----------+------------+-----------+
|transaction_id|worker_id| date_id|channel_id|market_id|    transaction_type|amount_local| amount_usd|transaction_outcome|is_fraud_flagged|is_disputed|is_reversed|velocity_score|fraud_loss_usd|processing_time_ms|channel_type|    channel_subtype|is_digital|avg_fraud_rate|requires_internet| full_date|year|    month|quarter|day_of_week|is_weekend|is_month_end|week_number|
+--------------+---------+--------+----------+---------+--------------------+------------+-----------+-------------------+----------------+-----------+-----------+--------------+--------------+------------------+------------+-------------------+----------+

In [16]:
# Left join: retain every record from the fact table
df_joined2 = df_joined1.join(
    df_worker,
    df_joined1["worker_id"] == df_worker["worker_id"],
    "left"
).drop(df_worker["worker_id"])  # avoids duplicate worker_id column

df_joined2.show(5)

+--------------+---------+--------+----------+---------+--------------------+------------+-----------+-------------------+----------------+-----------+-----------+--------------+--------------+------------------+------------+-------------------+----------+--------------+-----------------+----------+----+---------+-------+-----------+----------+------------+-----------+---------------+--------------------+-------------------+--------------------+-----------------+--------+-----------+---------+--------------------+
|transaction_id|worker_id| date_id|channel_id|market_id|    transaction_type|amount_local| amount_usd|transaction_outcome|is_fraud_flagged|is_disputed|is_reversed|velocity_score|fraud_loss_usd|processing_time_ms|channel_type|    channel_subtype|is_digital|avg_fraud_rate|requires_internet| full_date|year|    month|quarter|day_of_week|is_weekend|is_month_end|week_number|    worker_name|         gig_segment|account_tenure_days|            kyc_tier|           gender|age_band| ris

In [17]:
# Left join: retain every record from the fact table
df_joined3 = df_joined2.join(
    df_market,
    df_joined2["market_id"] == df_market["market_id"],
    "left"
).drop(df_market["market_id"])  # avoids duplicate market_id column

df_joined3.show(5)

+--------------+---------+--------+----------+---------+--------------------+------------+-----------+-------------------+----------------+-----------+-----------+--------------+--------------+------------------+------------+-------------------+----------+--------------+-----------------+----------+----+---------+-------+-----------+----------+------------+-----------+---------------+--------------------+-------------------+--------------------+-----------------+--------+-----------+---------+--------------------+------------+--------+--------------+------------------+-----------------+-----------+------------------+
|transaction_id|worker_id| date_id|channel_id|market_id|    transaction_type|amount_local| amount_usd|transaction_outcome|is_fraud_flagged|is_disputed|is_reversed|velocity_score|fraud_loss_usd|processing_time_ms|channel_type|    channel_subtype|is_digital|avg_fraud_rate|requires_internet| full_date|year|    month|quarter|day_of_week|is_weekend|is_month_end|week_number|    

## Write the final data

In [18]:
df_joined3.write.mode("overwrite").parquet(
    "output/fact_transactions_with_channel.parquet"
)

In [19]:
df_joined3.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/fact_transactions_with_channel")